# Phase 3 — Wind resource scoring

**Land-adjusted model:** Primary model uses the **core feature set** (honest OOF R²); extended weather-distribution features from notebook 01 are kept as a diagnostic ablation only.

Same three-layer scoring as solar (Phase 3 screening):

1. **Observed** — `wind_power_mean` from Phase 1
2. **Land-adjusted (OOF)** — location + NLCD + met + hub wind context (no `wind_power_mean` / hub-power proxy)
3. **Screening score** — percentile of observed wind power

| Input | Output |
|-------|--------|
| `cells_passing_gateway.parquet` | `wind_scores_by_cell.parquet` |
| | `models/wind_resource.joblib` |

In [1]:
N_CV_FOLDS = 5
RANDOM_STATE = 42
SPATIAL_BLOCK_DEG = 0.5

import json
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import geopandas as gpd
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import plotly.graph_objects as go
import xgboost as xgb
import yaml
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold

ROOT = Path('../..').resolve()
INPUT_PATH = ROOT / 'data' / 'processed' / 'cells_passing_gateway.parquet'
SCORES_PATH = ROOT / 'data' / 'processed' / 'wind_scores_by_cell.parquet'
CV_REPORT_PATH = ROOT / 'data' / 'processed' / 'wind_resource_cv_report.json'
MODEL_PATH = ROOT / 'models' / 'wind_resource.joblib'
CONFIG_PATH = ROOT / 'models' / 'wind_resource_config.yaml'
MAP_PATH = ROOT / 'maps' / 'wind_resource_ny.html'
NY_BOUNDARY_PATH = ROOT / 'data' / 'ref' / 'ny_state_boundary.geojson'

TARGET = 'wind_power_mean'
BASE_FEATURES = [
    'latitude', 'longitude', 'dist_to_transmission_km',
    'temperature_mean', 'surface_pressure_mean',
]
# notebook 01 wind distribution features (exclude target + v³ proxy)
WEATHER_R1_FEATURES = [
    'wind_speed_hub_mean', 'wind_speed_hub_std',
    'wind_hub_speed_p50', 'wind_hub_speed_p75', 'wind_hub_speed_p90', 'wind_hub_speed_p95',
    'wind_ramp_p95', 'weibull_k', 'weibull_A',
    'wind_speed_interannual_std', 'wind_speed_intraseasonal_std_mean',
    'wind_hub_speed_p50_DJF', 'wind_hub_speed_p90_JJA',
]

if not INPUT_PATH.exists():
    raise FileNotFoundError('Run 06_gateway_filter.ipynb first')

for p in [SCORES_PATH.parent, MODEL_PATH.parent, MAP_PATH.parent]:
    p.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)

ROOT: /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York


In [2]:
cells = pl.read_parquet(INPUT_PATH)
if 'in_ny_state' not in cells.columns:
    raise ValueError('Re-run 05_build_master_table.ipynb for in_ny_state')

pdf = cells.filter(pl.col('in_ny_state')).to_pandas()
pdf = pdf.drop(columns=['index_right'], errors='ignore')
print('Gateway-passing cells in NY:', len(pdf))

nlcd_features = [c for c in pdf.columns if c.startswith('frac_class_')]
weather_r1 = [c for c in WEATHER_R1_FEATURES if c in pdf.columns]
land_features_v1 = BASE_FEATURES + nlcd_features
land_features = BASE_FEATURES + weather_r1 + nlcd_features
print('R1 weather features used:', weather_r1)

model_df = pdf.dropna(subset=[TARGET] + land_features).copy()
model_df['spatial_block'] = (
    np.floor(model_df['latitude'] / SPATIAL_BLOCK_DEG) * 10_000
    + np.floor(model_df['longitude'] / SPATIAL_BLOCK_DEG)
).astype(int)
if 'county' not in model_df.columns:
    model_df['county'] = 'unknown'

print('Modeling rows:', len(model_df))

Gateway-passing cells in NY: 1089
R1 weather features used: ['wind_speed_hub_mean', 'wind_speed_hub_std', 'wind_hub_speed_p50', 'wind_hub_speed_p75', 'wind_hub_speed_p90', 'wind_hub_speed_p95', 'wind_ramp_p95', 'weibull_k', 'weibull_A', 'wind_speed_interannual_std', 'wind_speed_intraseasonal_std_mean', 'wind_hub_speed_p50_DJF', 'wind_hub_speed_p90_JJA']
Modeling rows: 1089


In [3]:
X_v1 = model_df[land_features_v1].astype(np.float32)
X_r1 = model_df[land_features].astype(np.float32)
y = model_df[TARGET].astype(np.float32)
groups = model_df['spatial_block'].values
gkf = GroupKFold(n_splits=min(N_CV_FOLDS, model_df['spatial_block'].nunique()))

xgb_params_v1 = dict(
    n_estimators=120, max_depth=4, learning_rate=0.08,
    subsample=0.85, colsample_bytree=0.85, random_state=RANDOM_STATE,
    n_jobs=4, objective='reg:squarederror',
)
xgb_params_r1 = dict(
    n_estimators=200, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    random_state=RANDOM_STATE, n_jobs=4, objective='reg:squarederror',
)

oof_land = np.zeros(len(model_df))
fold_rows = []
for fold, (tr, te) in enumerate(gkf.split(X_v1, y, groups)):
    m = xgb.XGBRegressor(**xgb_params_v1)
    m.fit(X_v1.iloc[tr], y.iloc[tr])
    pred = m.predict(X_v1.iloc[te])
    oof_land[te] = pred
    fold_rows.append({'fold': fold, 'r2': float(r2_score(y.iloc[te], pred))})
    print(f"Fold {fold}: R²={fold_rows[-1]['r2']:.3f}")

cv_land = {'r2': float(r2_score(y, oof_land)), 'mae': float(mean_absolute_error(y, oof_land))}
print('Land-adjusted OOF (v1 features):', cv_land)

oof_r1 = np.zeros(len(model_df))
for tr, te in gkf.split(X_r1, y, groups):
    m = xgb.XGBRegressor(**xgb_params_r1)
    m.fit(X_r1.iloc[tr], y.iloc[tr])
    oof_r1[te] = m.predict(X_r1.iloc[te])
cv_r1_r2 = float(r2_score(y, oof_r1))
print(f'Ablation R1 extended OOF R²={cv_r1_r2:.3f} | R1 ΔR²={cv_r1_r2 - cv_land["r2"]:.3f}')


Fold 0: R²=0.501


Fold 1: R²=0.522


Fold 2: R²=0.534


Fold 3: R²=0.513


Fold 4: R²=0.260
Land-adjusted OOF (v1 features): {'r2': 0.5063597685481106, 'mae': 9.561286427101157}


Ablation R1 extended OOF R²=0.987 | R1 ΔR²=0.480


In [4]:
final_model = xgb.XGBRegressor(**xgb_params_v1)
final_model.fit(X_v1, y)

observed = model_df[TARGET].values
obs_pct = pd.Series(observed).rank(pct=True).values

scores = model_df[['latitude', 'longitude', 'county', TARGET]].rename(
    columns={TARGET: 'resource_observed'}
)
scores['resource_observed_pct'] = obs_pct
scores['resource_land_adj_oof'] = oof_land
scores['resource_opportunity_oof'] = observed - oof_land
scores['resource_opportunity_pct'] = pd.Series(scores['resource_opportunity_oof']).rank(pct=True).values
scores['resource_screening_score'] = obs_pct

pl.from_pandas(scores).write_parquet(SCORES_PATH)
joblib.dump(final_model, MODEL_PATH)

config = {
    'target': TARGET, 'model_type': 'land_adjusted', 'features': land_features_v1,
    'feature_set': 'v1_hybrid_primary',
    'n_cells': len(model_df), 'cv_land_adjusted_oof': cv_land, 'cv_folds': fold_rows,
    'ablation_r1_extended_oof_r2': cv_r1_r2, 'r1_delta_r2': cv_r1_r2 - cv_land['r2'],
}
CV_REPORT_PATH.write_text(json.dumps(config, indent=2))
CONFIG_PATH.write_text(yaml.safe_dump(config))
print(f'Scores -> {SCORES_PATH}')

Scores -> /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/data/processed/wind_scores_by_cell.parquet


In [5]:
ny_boundary = gpd.read_file(NY_BOUNDARY_PATH)
outline = ny_boundary.boundary.iloc[0]
fig = go.Figure()
fig.add_trace(go.Scattermap(
    lat=scores['latitude'], lon=scores['longitude'], mode='markers',
    marker=dict(size=9, color=scores['resource_screening_score'], colorscale='Blues',
                showscale=True, opacity=0.65),
    hoverinfo='text',
    text=[f"obs={r.resource_observed:.1f}" for r in scores.itertuples()],
))
fig.add_trace(go.Scattermap(lat=list(outline.xy[1]), lon=list(outline.xy[0]),
    mode='lines', line=dict(width=2, color='#333'), hoverinfo='skip', showlegend=False))
fig.update_layout(map=dict(style='carto-positron', center=dict(lat=42.9, lon=-75.8), zoom=5.5),
    title='Wind screening score (observed wind power percentile)', height=800)
fig.write_html(str(MAP_PATH), include_plotlyjs='cdn')
print(f'Map -> {MAP_PATH}')

Map -> /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/maps/wind_resource_ny.html
